# پاکسازی داده‌های کووید‑۱۹ برای تحلیل آماری
این نوت‌بوک شامل گام‌های کامل پاکسازی (Data Cleaning) روی دیتاست جهانی کووید‑۱۹ است.  
مراحل انجام‌شده: حذف ستون‌های اضافی، مدیریت مقادیر گمشده، حذف رکوردهای نامعتبر، مهندسی ویژگی و استانداردسازی.

In [15]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [4]:
# در صورت نیاز مسیر فایل را اصلاح کنید
df = pd.read_csv('Crona_covid_19.csv')

# ایجاد یک کپی برای جلوگیری از تغییر در دیتای خام
df_clean = df.copy()
print("Data recive succsessfully")

Data recive succsessfully


## بررسی اولیه داده
در این مرحله، ساختار، نوع داده‌ها و تعداد مقادیر گمشده را بررسی می‌کنیم.

In [5]:
print("=== اطلاعات کلی ===")
df_clean.info()

print("\n=== ۵ رکورد اول ===")
display(df_clean.head())

print("\n=== تعداد مقادیر گمشده در هر ستون ===")
print(df_clean.isnull().sum())

=== اطلاعات کلی ===
<class 'pandas.DataFrame'>
RangeIndex: 231 entries, 0 to 230
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   active                  231 non-null    int64  
 1   activePerOneMillion     231 non-null    float64
 2   cases                   231 non-null    int64  
 3   casesPerOneMillion      231 non-null    int64  
 4   continent               229 non-null    str    
 5   country                 231 non-null    str    
 6   countryInfo             231 non-null    str    
 7   critical                231 non-null    int64  
 8   criticalPerOneMillion   231 non-null    float64
 9   deaths                  231 non-null    int64  
 10  deathsPerOneMillion     231 non-null    int64  
 11  oneCasePerPeople        231 non-null    int64  
 12  oneDeathPerPeople       231 non-null    int64  
 13  oneTestPerPeople        231 non-null    int64  
 14  population              231 non-n

,active,activePerOneMillion,cases,casesPerOneMillion,continent,country,countryInfo,critical,criticalPerOneMillion,deaths,...,oneTestPerPeople,population,recovered,recoveredPerOneMillion,tests,testsPerOneMillion,todayCases,todayDeaths,todayRecovered,updated
0,15098,370.46,234174,5746,Asia,Afghanistan,"{""_id"": 4, ""iso2"": ""AF"", ""iso3"": ""AFG"", ""lat"":...",0,0.0,7996,...,29,40754388,211080,5179.32,1390730,34125,0,0,0,1784464632992
1,1025,357.59,334863,116825,Europe,Albania,"{""_id"": 8, ""iso2"": ""AL"", ""iso3"": ""ALB"", ""lat"":...",0,0.0,3605,...,1,2866374,330233,115209.32,1941032,677173,0,0,0,1784464632985
2,82068,1809.65,272010,5998,Africa,Algeria,"{""_id"": 12, ""iso2"": ""DZ"", ""iso3"": ""DZA"", ""lat""...",0,0.0,6881,...,196,45350148,183061,4036.61,230960,5093,0,0,0,1784464632988
3,47850,617714.26,48015,619844,Europe,Andorra,"{""_id"": 20, ""iso2"": ""AD"", ""iso3"": ""AND"", ""lat""...",0,0.0,165,...,0,77463,0,0.00,249838,3225256,0,0,0,1784464633040
4,1971,56.27,107327,3064,Africa,Angola,"{""_id"": 24, ""iso2"": ""AO"", ""iso3"": ""AGO"", ""lat""...",0,0.0,1937,...,23,35027343,103419,2952.52,1499795,42818,0,0,0,1784464633013



=== تعداد مقادیر گمشده در هر ستون ===
active                    0
activePerOneMillion       0
cases                     0
casesPerOneMillion        0
continent                 2
country                   0
countryInfo               0
critical                  0
criticalPerOneMillion     0
deaths                    0
deathsPerOneMillion       0
oneCasePerPeople          0
oneDeathPerPeople         0
oneTestPerPeople          0
population                0
recovered                 0
recoveredPerOneMillion    0
tests                     0
testsPerOneMillion        0
todayCases                0
todayDeaths               0
todayRecovered            0
updated                   0
dtype: int64


## حذف ستون‌های اضافی
ستون‌هایی که مقدار ثابت (صفر) دارند، یا اطلاعات تکراری/غیرعددی هستند (مانند countryInfo و نسبت‌های مشتق‌شده) را حذف می‌کنیم تا از چندهمخطی جلوگیری شود.

In [6]:
cols_to_drop = [
    'updated',          # تایم‌استمپ یکسان
    'todayCases',       # همه صفر
    'todayDeaths',      # همه صفر
    'todayRecovered',   # همه صفر
    'activePerOneMillion',
    'casesPerOneMillion',
    'criticalPerOneMillion',
    'deathsPerOneMillion',
    'recoveredPerOneMillion',
    'testsPerOneMillion',
    'oneCasePerPeople',
    'oneDeathPerPeople',
    'oneTestPerPeople'
]

df_clean.drop(columns=cols_to_drop, inplace=True, errors='ignore')
print(f"ستون‌های باقیمانده: {list(df_clean.columns)}")

ستون‌های باقیمانده: ['active', 'cases', 'continent', 'country', 'countryInfo', 'critical', 'deaths', 'population', 'recovered', 'tests']


## مدیریت Missing Values
- رکوردهایی که `continent` ندارند (مانند کشتی‌های گردشگری) برای تحلیل بین‌کشوری بی‌کاربرد هستند و حذف می‌شوند.
- ستون‌های `recovered` و `critical` که پراکندگی گمشدگی بالایی دارند، با عدد صفر پر می‌شوند (فرض بر عدم گزارش یا صفر بودن).

In [7]:
# حذف رکوردهای بدون قاره
df_clean.dropna(subset=['continent'], inplace=True)

# پر کردن recovered و critical با صفر (چون در بسیاری از کشورها گزارش نشده)
df_clean[['recovered', 'critical']] = df_clean[['recovered', 'critical']].fillna(0)

print(f"تعداد رکوردهای پس از حذف رکوردهای فاقد قاره: {len(df_clean)}")
print("مقادیر گمشده باقیمانده:")
print(df_clean.isnull().sum())

تعداد رکوردهای پس از حذف رکوردهای فاقد قاره: 229
مقادیر گمشده باقیمانده:
active         0
cases          0
continent      0
country        0
countryInfo    0
critical       0
deaths         0
population     0
recovered      0
tests          0
dtype: int64


## فیلتر کردن رکوردهای نامعتبر
- کشورهایی با جمعیت کمتر از ۱۰۰۰ نفر (اغلب جزایر بسیار کوچک یا خطاهای داده) حذف می‌شوند.
- برای متغیر `cases` از روش **دامنه بین‌چهارکی (IQR)** برای حذف نقاط پرت شدید استفاده می‌شود تا تحلیل آماری پایدارتر شود.

In [9]:
# فیلتر جمعیت
df_clean = df_clean[df_clean['population'] > 1000]

# حذف پرت‌های شدید با استفاده از IQR برای ستون cases
Q1 = df_clean['cases'].quantile(0.25)
Q3 = df_clean['cases'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 3 * IQR   # استفاده از 3IQR برای شناسایی پرت‌های خیلی شدید
upper_bound = Q3 + 3 * IQR

df_clean = df_clean[(df_clean['cases'] >= lower_bound) & (df_clean['cases'] <= upper_bound)]

print(f"تعداد رکوردها پس از حذف ‌: {len(df_clean)}")

تعداد رکوردها پس از حذف ‌: 189


## مهندسی ویژگی
برای تحلیل‌های آماری، متغیرهای نسبت‌وار زیر را می‌سازیم:
- **نرخ مرگ‌ومیر (CFR)**: `deaths / cases`
- **نرخ بهبودی**: `recovered / cases`
- **نرخ موارد فعال**: `active / cases`
- **تعداد تست به ازای هر نفر**: `tests / population`

(در صورت صفر بودن مخرج، مقدار `NaN` یا بینهایت با `0` جایگزین می‌شود)

In [10]:
# جایگزینی مقادیر بینهایت و NaN ناشی از تقسیم بر صفر با 0
df_clean['cfr'] = df_clean['deaths'].div(df_clean['cases']).replace([np.inf, -np.inf], np.nan).fillna(0)
df_clean['recovery_rate'] = df_clean['recovered'].div(df_clean['cases']).replace([np.inf, -np.inf], np.nan).fillna(0)
df_clean['active_rate'] = df_clean['active'].div(df_clean['cases']).replace([np.inf, -np.inf], np.nan).fillna(0)
df_clean['tests_per_capita'] = df_clean['tests'].div(df_clean['population']).replace([np.inf, -np.inf], np.nan).fillna(0)

print("متغیرهای جدید ساخته شدند:")
print(df_clean[['cfr', 'recovery_rate', 'active_rate', 'tests_per_capita']].describe())

متغیرهای جدید ساخته شدند:
              cfr  recovery_rate  active_rate  tests_per_capita
count  189.000000     189.000000   189.000000        189.000000
mean     0.013246       0.705486     0.281268          1.835599
std      0.017700       0.418519     0.420963          3.411738
min      0.000000       0.000000     0.000000          0.000000
25%      0.004445       0.228490     0.000862          0.062198
50%      0.008749       0.971350     0.010598          0.618038
75%      0.017535       0.988802     0.767605          1.930391
max      0.180745       1.000000     1.000000         22.165247


## استانداردسازی (Standardization)
برای آزمون‌های آماری مانند رگرسیون و ANOVA، متغیرهای عددی اصلی (به جز نسبت‌ها) را استاندارد می‌کنیم تا میانگین ۰ و واریانس ۱ داشته باشند.
> **توجه**: اگر قصد تفسیر ضرایب به صورت خام را دارید، این سلول را اجرا نکنید.

In [16]:
num_cols = ['active', 'cases', 'critical', 'deaths', 'population', 'recovered', 'tests']

# حذف رکوردهای حاوی NaN در این ستون‌ها (در صورت وجود) برای جلوگیری از خطای اسکیلر
df_clean_scaled = df_clean.dropna(subset=num_cols).copy()

scaler = StandardScaler()
df_clean_scaled[num_cols] = scaler.fit_transform(df_clean_scaled[num_cols])

print("آمار متغیرهای استانداردشده (میانگین باید نزدیک صفر باشد):")
print(df_clean_scaled[num_cols].mean().round(2))
print(df_clean_scaled[num_cols].std().round(2))

# در صورت تمایل، جایگزین دیتافریم اصلی با نسخه‌ی استاندارد کنید
df_final = df_clean_scaled

آمار متغیرهای استانداردشده (میانگین باید نزدیک صفر باشد):
active       -0.0
cases         0.0
critical     -0.0
deaths       -0.0
population   -0.0
recovered     0.0
tests         0.0
dtype: float64
active        1.0
cases         1.0
critical      1.0
deaths        1.0
population    1.0
recovered     1.0
tests         1.0
dtype: float64


## ذخیره‌سازی خروجی
داده‌های پاک‌شده و آماده را در یک فایل CSV جدید ذخیره می‌کنیم تا در تحلیل‌های بعدی از آن استفاده کنیم.

In [17]:
df_final.to_csv('covid19_cleaned_final.csv', index=False)
print("فایل 'covid19_cleaned_final.csv' با موفقیت ذخیره شد.")
print(f"شکل نهایی داده‌ها: {df_final.shape}")

فایل 'covid19_cleaned_final.csv' با موفقیت ذخیره شد.
شکل نهایی داده‌ها: (189, 14)


In [18]:
print("\n=== خلاصه آماری داده‌های نهایی ===")
display(df_final.describe())

print("\n=== بررسی مجدد مقادیر گمشده ===")
print(df_final.isnull().sum().sum())


=== خلاصه آماری داده‌های نهایی ===


,active,cases,critical,deaths,population,recovered,tests,cfr,recovery_rate,active_rate,tests_per_capita
count,1.890000e+02,1.890000e+02,1.890000e+02,1.890000e+02,1.890000e+02,1.890000e+02,1.890000e+02,189.000000,189.000000,189.000000,189.000000
mean,-4.640615e-17,4.699357e-17,-9.398713e-18,-1.879743e-17,-4.699357e-18,5.169292e-17,7.049035e-18,0.013246,0.705486,0.281268,1.835599
std,1.002656e+00,1.002656e+00,1.002656e+00,1.002656e+00,1.002656e+00,1.002656e+00,1.002656e+00,0.017700,0.418519,0.420963,3.411738
min,-3.111863e-01,-6.522952e-01,-2.274319e-01,-5.980853e-01,-1.918348e-01,-5.337659e-01,-2.926186e-01,0.000000,0.000000,0.000000,0.000000
25%,-3.109570e-01,-6.232089e-01,-2.274319e-01,-5.782920e-01,-1.891640e-01,-5.313066e-01,-2.844540e-01,0.004445,0.228490,0.000862,0.062198
50%,-3.072563e-01,-5.200008e-01,-2.274319e-01,-4.725639e-01,-1.606862e-01,-4.820767e-01,-2.507005e-01,0.008749,0.971350,0.010598,0.618038
75%,-2.508100e-01,2.653767e-01,-2.274319e-01,1.218495e-01,-8.122043e-02,-1.729077e-02,-1.060264e-01,0.017535,0.988802,0.767605,1.930391
max,6.814741e+00,4.052445e+00,9.811440e+00,5.431860e+00,1.316968e+01,4.331064e+00,8.920584e+00,0.180745,1.000000,1.000000,22.165247



=== بررسی مجدد مقادیر گمشده ===
0
